In [ ]:
!pip install -U transformers

# !pip install av tqdm trl peft
!pip install timm dlib tqdm gdown
!wget https://github.com/italojs/facial-landmarks-recognition/raw/refs/heads/master/shape_predictor_68_face_landmarks.dat
!wget https://megc2026.github.io/files/me_vqa_samm_casme2_smic_v2.jsonl

!gdown 13bpZLp7659VPkhaD-mwh8uufol6kX44p
!gdown 1EojrW6morT0duZ1G0LWAwMy-eUjo_3TV

from IPython.display import clear_output
clear_output(wait=False)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch import optim


from transformers import CLIPProcessor, CLIPModel
import gc
import matplotlib.pyplot as plt
# from huggingface_hub import hf_hub_download
# from sklearn.model_selection import GroupShuffleSplit

import re
import json
import os
from tqdm import tqdm
import timm
import dlib
from glob import glob

import random
from collections import defaultdict
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, recall_score


# PREPROCESSING

In [ ]:
# đọc jsonl -> merge coarse vao dataframe
data = []
with open("/kaggle/working/me_vqa_samm_casme2_smic_v2.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

json_df = pd.DataFrame(data)

coarse_df = json_df[
    (json_df["dataset"] == "casme2") &
    (json_df["question"] == "What is the coarse expression class?")
][["filename", "subject", "answer"]]
coarse_df = coarse_df.rename(columns={"answer": "coarse emotion", "filename":"Filename", "subject":"Subject"})
coarse_df["Subject"] = coarse_df["Subject"].str.replace("sub", "").astype(int)

df = pd.read_excel("/kaggle/input/datasets/tramanh1511tn/casme-dataset/CASME2-coding-20140508.xlsx")
df = df[pd.to_numeric(df["ApexFrame"], errors="coerce").notna()]
print(df.shape)
# df.head()
df = df.merge(
    coarse_df,
    on=["Filename", "Subject"],
    how="left"
)
df = df.dropna(subset=["coarse emotion"], how="all")
df = df.reset_index(drop=True)
# print(df["Estimated Emotion"].value_counts())
df

In [ ]:
coarse_df_samm = json_df[
    (json_df["dataset"] == "samm") &
    (json_df["question"] == "What is the coarse expression class?")
][["filename", "subject", "answer"]]
coarse_df_samm = coarse_df_samm.rename(columns={"answer": "coarse emotion", "filename":"Filename", "subject":"Subject"})
coarse_df_samm["Subject"] = coarse_df_samm["Subject"].str.replace("sub", "").astype(int)

df_samm = pd.read_excel("/kaggle/input/datasets/khuongvutramanh/samm-data-qq/SAMM_Micro_FACS_Codes_v2.xlsx")
df_samm = df_samm[pd.to_numeric(df_samm["Apex Frame"], errors="coerce").notna()]
print(df_samm.shape)
# df.head()
df_samm = df_samm.merge(
    coarse_df_samm,
    on=["Filename", "Subject"],
    how="left"
)
df_samm["Estimated Emotion"] = df_samm["Estimated Emotion"].str.lower()
df_samm = df_samm.dropna(subset=["coarse emotion"], how="all")
df_samm = df_samm.reset_index(drop=True)
df_samm = df_samm.rename(columns={
    "Onset Frame": "OnsetFrame", 
    "Apex Frame": "ApexFrame"
})

print(df_samm["Estimated Emotion"].value_counts())
df_samm

In [ ]:
from collections import defaultdict
import os, re, json

def norm_text(s):
    s = str(s).strip().lower().replace("\\", "/")
    s = re.sub(r"/+", "/", s)
    s = re.sub(r"\.(jpg|jpeg|png|bmp)$", "", s)
    return s

def get_basename_noext(s):
    s = norm_text(s)
    return os.path.basename(s)

def extract_smic_subject_seq(s):
    """
    Tách subject và seq từ các dạng:
    - s1_ne_01
    - smic/s1/micro/negative/s1_ne_01
    - HS/s1/micro/negative/s1_ne_01
    """
    s = norm_text(s)

    subject = None
    seq = None

    m_sub = re.search(r'(^|/)(s\d+)(/|$)', s)
    if m_sub:
        subject = m_sub.group(2)

    m_seq = re.search(r'(s\d+_[a-z]{2}_\d+)', s)
    if m_seq:
        seq = m_seq.group(1)
    else:
        base = get_basename_noext(s)
        if re.match(r's\d+_[a-z]{2}_\d+$', base):
            seq = base

    return subject, seq

# ===== MAIN QA INDEX =====
qa_dict_full = defaultdict(list)       # exact dataset::normalized_id
qa_dict_base = defaultdict(list)       # dataset::basename
qa_dict_smic_pair = defaultdict(list)  # ("smic", subject, seq)

all_smic_keys = []

with open("/kaggle/working/me_vqa_samm_casme2_smic_v2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        item = json.loads(line)
        dataset = str(item.get("dataset", "")).strip().lower()
        if dataset not in ["casme2", "samm", "smic"]:
            continue

        image_id = item.get("image_id", "")
        image_id_norm = norm_text(image_id)
        base = get_basename_noext(image_id_norm)

        qa_item = {
            "question": item["question"],
            "answer": item["answer"]
        }

        qa_dict_full[f"{dataset}::{image_id_norm}"].append(qa_item)
        qa_dict_base[f"{dataset}::{base}"].append(qa_item)

        if dataset == "smic":
            sub, seq = extract_smic_subject_seq(image_id_norm)
            if sub is not None and seq is not None:
                qa_dict_smic_pair[(dataset, sub, seq)].append(qa_item)
            all_smic_keys.append(image_id_norm)

print("QA index loaded.")
print("Exact keys:", len(qa_dict_full))
print("Base keys :", len(qa_dict_base))
print("SMIC pair :", len(qa_dict_smic_pair))
print("SMIC raw keys:", len(all_smic_keys))


def dedup_qa_list(lst):
    seen = set()
    out = []
    for x in lst:
        key = (x["question"], x["answer"])
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out


def get_qa_list(dataset, *candidate_ids, smic_subject=None, smic_seq=None):
    dataset = str(dataset).lower()
    hits = []

    # 1) exact full-key
    for cid in candidate_ids:
        cid_norm = norm_text(cid)
        hits.extend(qa_dict_full.get(f"{dataset}::{cid_norm}", []))

    if hits:
        return dedup_qa_list(hits)

    # 2) basename
    for cid in candidate_ids:
        base = get_basename_noext(cid)
        hits.extend(qa_dict_base.get(f"{dataset}::{base}", []))

    if hits:
        return dedup_qa_list(hits)

    # 3) riêng cho SMIC: pair(subject, seq)
    if dataset == "smic" and smic_subject is not None and smic_seq is not None:
        hits.extend(qa_dict_smic_pair.get(("smic", norm_text(smic_subject), norm_text(smic_seq)), []))
        if hits:
            return dedup_qa_list(hits)

        # 4) fuzzy contains cho SMIC
        subject_norm = norm_text(smic_subject)
        seq_norm = norm_text(smic_seq)

        fuzzy_keys = [
            k for k in all_smic_keys
            if (seq_norm in k) and (subject_norm in k)
        ]

        for k in fuzzy_keys:
            hits.extend(qa_dict_full.get(f"smic::{k}", []))

        if hits:
            return dedup_qa_list(hits)

    return []

## OPTICAL FLOW

In [ ]:
def compute_flow(onset_img, apex_img):
    onset_img = cv2.resize(onset_img, (224, 224))
    apex_img = cv2.resize(apex_img, (224, 224))
    
    gray1 = cv2.cvtColor(onset_img, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(apex_img, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        gray1, gray2,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0
    )

    return flow

def flow_to_rgb(flow):
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    # hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
    hsv = np.zeros((224, 224, 3), dtype=np.uint8)
    hsv[..., 0] = angle * 180 / np.pi / 2
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


    # flow_rgb = np.zeros((flow_x.shape[0], flow_x.shape[1], 3), dtype=np.uint8)
    # flow_rgb[..., 0] = flow_x_rescaled  # Gán flow_x cho kênh đỏ
    # flow_rgb[..., 1] = flow_y_rescaled  # Gán flow_y cho kênh xanh
    # flow_rgb[..., 2] = 128  # Giá trị cố định cho kênh xanh dương (tùy chỉnh)

    return rgb

# lấy optical flow + chuyển thành ảnh rgb
def get_optical_flow_image(onset_img, apex_img):
    flow = compute_flow(onset_img, apex_img)
    flow_rgb = flow_to_rgb(flow)
    
    return flow_rgb

In [ ]:
# test optical flow
onset = cv2.imread("/kaggle/input/datasets/tramanh1511tn/casme-dataset/Cropped/Cropped/sub01/EP02_01f/reg_img46.jpg")
apex = cv2.imread("/kaggle/input/datasets/tramanh1511tn/casme-dataset/Cropped/Cropped/sub01/EP02_01f/reg_img59.jpg")

flow_img = get_optical_flow_image(onset, apex)

# img_rgb = cv2.cvtColor(flow_img, cv2.COLOR_BGR2RGB)

plt.imshow(flow_img)
plt.axis("off")
plt.title("Optical flow")
plt.show()

## AU EXTRACTION

In [ ]:
AU_MAPPING = {
    1: {"name": "inner brow raiser",    "landmarks": [20, 21, 22, 23]},
    2: {"name": "outer brow raiser",    "landmarks": [17, 18, 19, 24, 25, 26]},
    4: {"name": "brow lowerer",         "landmarks": [19, 20, 21, 22, 23, 24, 27]},
    5: {"name": "upper lid raiser",     "landmarks": [37, 38, 43, 44]},            
    6: {"name": "cheek raiser",         "landmarks": [39, 40, 41, 45, 46, 47]},
    7: {"name": "lid tightener",        "landmarks": list(range(36, 48))},
    9: {"name": "nose wrinkler",        "landmarks": list(range(27, 36))},
    10: {"name": "upper lip raiser",    "landmarks": [49, 50, 51, 52, 53]},
    12: {"name": "lip corner puller",   "landmarks": [48, 54]},
    14: {"name": "dimpler",             "landmarks": [48, 54]},
    15: {"name": "lip corner depressor","landmarks": [48, 54, 56, 57, 58]},
    17: {"name": "chin raiser",         "landmarks": [6, 7, 8, 9, 10, 55, 56, 57, 58, 59]},
    20: {"name": "lip stretcher",       "landmarks": [48, 54]},
    23: {"name": "lip tightener",       "landmarks": list(range(48, 68))},
    24: {"name": "lip pressor",         "landmarks": list(range(48, 68))},
    25: {"name": "lip part",            "landmarks": list(range(60, 68))},
    26: {"name": "jaw drop",            "landmarks": list(range(48, 68))}
}

def get_name_au_label(raw_au_string):
    extracted_numbers = re.findall(r'\d+', raw_au_string)
    au_numbers = []
    for num in extracted_numbers:
        n = int(num)
        if n not in au_numbers:
            au_numbers.append(n)

    au_names = []
    for num in au_numbers:
        if num in AU_MAPPING:
            name = AU_MAPPING[num]["name"]
            au_names.append(name)

    return au_names

def extract_au_roi_by_indices(image, landmarks, point_indices, padding_ratio=0.15, target_size=(224, 224)):
    points = landmarks[point_indices]
    
    x_min, x_max = np.min(points[:, 0]), np.max(points[:, 0])
    y_min, y_max = np.min(points[:, 1]), np.max(points[:, 1])
    
    face_width = landmarks[16][0] - landmarks[0][0]
    pad = int(face_width * padding_ratio)

    box_height = y_max - y_min
    box_width = x_max - x_min
    if box_height < pad:
        y_min -= pad // 2; y_max += pad // 2
    if box_width < pad:
        x_min -= pad // 2; x_max += pad // 2

    start_x = max(0, int(x_min - pad))
    start_y = max(0, int(y_min - pad))
    end_x = min(image.shape[1], int(x_max + pad))
    end_y = min(image.shape[0], int(y_max + pad))

    roi = image[start_y:end_y, start_x:end_x]
    
    if roi.size == 0:
        return np.zeros((target_size[1], target_size[0], 3), dtype=np.uint8)
        
    img = cv2.resize(roi, target_size, interpolation=cv2.INTER_AREA)
    return img


def process_roi_with_raw_au(raw_au_string, image, landmarks):
    if landmarks is None:
        print("Landmarks detection failed")
        return [], {}

    
    # trích xuất các số AU từ chuỗi (bỏ qua L, R)
    extracted_numbers = re.findall(r'\d+', raw_au_string)
    # au_numbers = list(set([int(num) for num in extracted_numbers]))
    au_numbers = []
    for num in extracted_numbers:
        n = int(num)
        if n not in au_numbers:
            au_numbers.append(n)
    
    au_names = []
    au_rois = {}
    
    # 2. Lấy dữ liệu từ Dictionary đồng nhất
    for num in au_numbers:
        if num in AU_MAPPING:
            name = AU_MAPPING[num]["name"]
            au_names.append(name)
            
            # Lấy tọa độ để đưa vào hàm extract_au_roi
            point_indices = AU_MAPPING[num]["landmarks"]

            # Truyền mảng indices thay vì truyền tên để hàm crop hoạt động
            roi_img = extract_au_roi_by_indices(image, landmarks, point_indices)
            au_rois[name] = roi_img
        else:
            print(f"AU chưa được định nghĩa: AU{num}")
    return au_names, au_rois

def process_all_roi(image, landmarks):    
    au_names = []
    au_rois = {}
    au_numbers = sorted(AU_MAPPING.keys())
    
    # 2. Lấy dữ liệu từ Dictionary đồng nhất
    for num in au_numbers:
        if num in AU_MAPPING:
            name = AU_MAPPING[num]["name"]
            au_names.append(name)
            
            # Lấy tọa độ để đưa vào hàm extract_au_roi
            point_indices = AU_MAPPING[num]["landmarks"]

            # Truyền mảng indices thay vì truyền tên để hàm crop hoạt động
            roi_img = extract_au_roi_by_indices(image, landmarks, point_indices)
            au_rois[name] = roi_img
        else:
            print(f"AU chưa được định nghĩa: AU{num}")
    return au_names, au_rois

In [ ]:
ALL_AU_NAME =  []
for au_num in AU_MAPPING:
    ALL_AU_NAME.append(AU_MAPPING[au_num]["name"])

print(len(ALL_AU_NAME))
print(ALL_AU_NAME)


ALL_EMOTION = ["happiness", "surprise", "fear", "disgust", "anger", "sadness", "repression"]
ALL_COARSE = ["positive", "negative", "surprise"]

In [ ]:
# def get_au_roi(image, landmarks, au_name, scale=1.5):
#     if au_name not in AU_LANDMARK_MAP:
#         raise ValueError(f"AU '{au_name}' không tồn tại trong từ điển mapping!")

#     indices = AU_LANDMARK_MAP[au_name]
#     pts = landmarks[indices]

#     x, y, w, h = cv2.boundingRect(pts)

#     # expand bounding box
#     cx, cy = x + w // 2, y + h // 2
#     w = int(w * scale)
#     h = int(h * scale)

#     x = max(cx - w // 2, 0)
#     y = max(cy - h // 2, 0)

#     roi = image[y:y+h, x:x+w]

#     if roi.size == 0:
#         return None

#     return cv2.resize(roi, (224, 224))

In [ ]:
class AUExtractor:
    def __init__(self, predictor_path):
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor(predictor_path)
        # self.roi_mapping = {
        #     "chin": (6,11),
        #     "left_eyebrow": (17, 22),   # AU1, AU2, AU4
        #     "right_eyebrow": (22, 27),  # AU1, AU2, AU4
        #     "nose": (27,36),
        #     "left_eye": (36, 42),       # AU6, AU7
        #     "right_eye": (42, 48),      # AU6, AU7
        #     "mouth": (48, 68)           # AU12, AU15, AU17,...
        # }

    def extract_landmarks(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray)

        if len(faces) == 0:
            return None
        
        shape = self.predictor(gray, faces[0])
        coords = np.array([[p.x, p.y] for p in shape.parts()])
        return coords

    # căn chỉnh lại face bị nghiêng dựa vào 2 mắt
    # def align_face(self, image, landmarks):
    #     left_eye_pts = landmarks[36:42]
    #     right_eye_pts = landmarks[42:48]

    #     left_eye_center = left_eye_pts.mean(axis=0).astype("int")
    #     right_eye_center = right_eye_pts.mean(axis=0).astype("int")

    #     dY = right_eye_center[1] - left_eye_center[1]
    #     dX = right_eye_center[0] - left_eye_center[0]
    #     angle = np.degrees(np.arctan2(dY, dX))

    #     eyes_center = (int((left_eye_center[0] + right_eye_center[0]) // 2),
    #                    int((left_eye_center[1] + right_eye_center[1]) // 2))

    #     # Xoay ảnh và landmarks
    #     M = cv2.getRotationMatrix2D(eyes_center, angle, scale=1.0)
    #     aligned_img = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]), flags=cv2.INTER_CUBIC)
        
    #     aligned_landmarks = np.zeros_like(landmarks)
    #     for i, (x, y) in enumerate(landmarks):
    #         pt = np.array([x, y, 1.0])
    #         aligned_pt = np.dot(M, pt)
    #         aligned_landmarks[i] = (int(aligned_pt[0]), int(aligned_pt[1]))

    #     return aligned_img, aligned_landmarks
            
        # return rois

In [ ]:
# test AU
def visualize():
    # df = pd.read_excel("/kaggle/input/datasets/tramanh1511tn/casme-dataset/CASME2-coding-20140508.xlsx")
    # df = df[pd.to_numeric(df["ApexFrame"], errors="coerce").notna()]
    # print(df.shape)

    # lấy 1 mẫu visual test
    # sample = df.iloc[33]
    sample = df.iloc[22]
    
    print(sample)

    subject = sample["Subject"]
    filename = sample["Filename"]
    apex = int(sample["ApexFrame"])
    au_raw = str(sample["Action Units"])
    sub_folder = f"sub{subject:02d}"
    root_dir = "/kaggle/input/datasets/tramanh1511tn/casme-dataset/Cropped/Cropped"
    apex_path = os.path.join(root_dir, sub_folder, filename, f"reg_img{apex}.jpg")
    # root_dir = "/kaggle/input/datasets/tramanh1511tn/casme-dataset/CASME2_RAW_selected/CASME2_RAW_selected/CASME2_RAW_selected"
    # apex_path = os.path.join(root_dir, sub_folder, filename, f"img{apex}.jpg")

    au_extractor = AUExtractor("/kaggle/working/shape_predictor_68_face_landmarks.dat")
    image = cv2.imread(apex_path)    
    landmarks = au_extractor.extract_landmarks(image)
    au_names, au_rois = process_roi_with_raw_au(au_raw, image, landmarks)
    print("AU name:", au_names)
    num_rois = len(au_rois)
        
    if num_rois == 0:
        print("Không có ROI nào được tạo ra.")
    else:
        fig, axes = plt.subplots(1, num_rois + 1, figsize=(4 * (num_rois + 1), 4))
        if num_rois == 0: 
            axes = [axes]
            
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        for (x, y) in landmarks:
            cv2.circle(image_rgb, (x, y), 2, (0, 255, 0), -1)
                
        axes[0].imshow(image_rgb)
        axes[0].set_title("Apex frame & Landmarks")
        axes[0].axis("off")

            # 2. Hiển thị lần lượt các ROI
        for idx, (name, roi_img) in enumerate(au_rois.items()):
                # Chuyển BGR sang RGB cho ROI
            roi_rgb = cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB)
                
                # Vẽ lên subplot tương ứng (bỏ qua ô đầu tiên của ảnh gốc)
            ax = axes[idx + 1]
            ax.imshow(roi_rgb)
            ax.set_title(f"ROI: {name}")
            ax.axis("off")

        plt.tight_layout()
        plt.show()


visualize()

## CREATE SAMPLE

In [ ]:

# !wget https://megc2026.github.io/files/me_vqa_samm_casme2_smic_v2.jsonl

# df = pd.read_excel("/kaggle/input/datasets/tramanh1511tn/casme-dataset/CASME2-coding-20140508.xlsx")
# df = df[pd.to_numeric(df["ApexFrame"], errors="coerce").notna()]
# print(df.shape)

# def merge_vqa_info(jsonl_path, df_original):
#     au_mapping = {}
    
#     with open(jsonl_path, 'r', encoding='utf-8') as f:
#         for line in f:
#             item = json.loads(line)
#             question = item.get("question", "").strip()
#             filename = item.get("filename", "")
#             answer = item.get("answer", "")
            
#             # Lọc các câu hỏi lấy danh sách Action Units
#             if question in ["What is the action unit?", "What are the action units?"]:
#                 au_mapping[filename] = answer

#     df_au = pd.DataFrame(list(au_mapping.items()), columns=['Filename', 'AU'])
#     df_merged = pd.merge(df_original, df_au, on='Filename', how='left')
#     return df_merged

# df_au = merge_vqa_info('me_vqa_samm_casme2_smic_v2.jsonl', df)
# df_au = df_au[df_au['AU'].notna()]
# print(df_au.shape)
# df_au.head()

In [ ]:
# root_dir = "/kaggle/input/datasets/tramanh1511tn/casme-dataset/Cropped/Cropped"
# out_dir = "megc_processed"

from glob import glob

def natural_key(path):
    base = os.path.basename(path)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', base)]

def list_image_files(folder):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.JPG", "*.JPEG", "*.PNG", "*.BMP"]
    paths = []
    for ext in exts:
        paths.extend(glob(os.path.join(folder, ext)))
    return sorted(paths, key=natural_key)

def detect_apex_frame_from_sequence(frame_paths):
    if len(frame_paths) < 2:
        return 0

    onset = cv2.imread(frame_paths[0])
    if onset is None:
        return 0

    onset = cv2.resize(onset, (224, 224))
    onset_gray = cv2.cvtColor(onset, cv2.COLOR_BGR2GRAY)

    best_idx = 1
    best_score = -1.0

    for i in range(1, len(frame_paths)):
        img = cv2.imread(frame_paths[i])
        if img is None:
            continue
        img = cv2.resize(img, (224, 224))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        score = cv2.absdiff(gray, onset_gray).mean()

        if score > best_score:
            best_score = score
            best_idx = i

    return best_idx


def build_smic_samples(root_dir, out_dir):
    data_infos = []
    au_extractor = AUExtractor("/kaggle/working/shape_predictor_68_face_landmarks.dat")

    n_total = 0
    n_hit_qa = 0
    n_skip_qa = 0

    subject_dirs = sorted(
        [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))],
        key=natural_key
    )

    for subject_dir in tqdm(subject_dirs, desc="Create sample [smic]"):
        subject_path = os.path.join(root_dir, subject_dir)
        micro_dir = os.path.join(subject_path, "micro")

        if not os.path.isdir(micro_dir):
            continue

        class_dirs = sorted(
            [d for d in os.listdir(micro_dir) if os.path.isdir(os.path.join(micro_dir, d))],
            key=natural_key
        )

        for coarse in class_dirs:
            coarse_lower = coarse.lower()
            class_path = os.path.join(micro_dir, coarse)

            seq_dirs = sorted(
                [d for d in os.listdir(class_path) if os.path.isdir(os.path.join(class_path, d))],
                key=natural_key
            )

            for seq_name in seq_dirs:
                n_total += 1
                seq_path = os.path.join(class_path, seq_name)
                frame_paths = list_image_files(seq_path)

                if len(frame_paths) < 2:
                    continue

                onset_path = frame_paths[0]
                apex_idx = detect_apex_frame_from_sequence(frame_paths)
                apex_path = frame_paths[apex_idx]

                # Tạo nhiều candidate ID nhất có thể
                rel_path = os.path.relpath(seq_path, root_dir)  # s1/micro/negative/s1_ne_01
                qa_candidates = [
                    seq_name,
                    rel_path,
                    os.path.join(subject_dir, "micro", coarse, seq_name),
                    os.path.join("hs", subject_dir, "micro", coarse, seq_name),
                    os.path.join("smic", subject_dir, "micro", coarse, seq_name),
                    os.path.join("smic_all_cropped", subject_dir, "micro", coarse, seq_name),
                    f"{subject_dir}_{seq_name}",
                    f"{subject_dir}/{seq_name}",
                ]

                qa_list = get_qa_list(
                    "smic",
                    *qa_candidates,
                    smic_subject=subject_dir,
                    smic_seq=seq_name
                )

                if len(qa_list) == 0:
                    n_skip_qa += 1
                    continue

                n_hit_qa += 1

                onset = cv2.imread(onset_path)
                apex = cv2.imread(apex_path)

                if onset is None or apex is None:
                    continue

                flow_img = get_optical_flow_image(onset, apex)

                landmarks = au_extractor.extract_landmarks(apex)
                if landmarks is None:
                    continue

                # giữ nguyên strategy ROI hiện tại của bạn
                au_names, roi_imgs = process_all_roi(apex, landmarks)
                if not roi_imgs or len(roi_imgs) == 0:
                    continue

                subject_id = f"smic_{subject_dir}"
                sample_id = os.path.join(subject_id, coarse_lower, seq_name)
                sample_out_dir = os.path.join(out_dir, sample_id)
                os.makedirs(sample_out_dir, exist_ok=True)

                out_apex_path = os.path.join(sample_out_dir, "apex.jpg")
                out_flow_path = os.path.join(sample_out_dir, "flow.jpg")

                cv2.imwrite(out_apex_path, apex)
                cv2.imwrite(out_flow_path, flow_img)

                out_roi_paths = []
                for i, (name, roi_img) in enumerate(roi_imgs.items()):
                    roi_path = os.path.join(sample_out_dir, f"roi_{i+1:02d}.jpg")
                    cv2.imwrite(roi_path, roi_img)
                    out_roi_paths.append(roi_path)

                data_infos.append({
                    "dataset": "smic",
                    "subject": subject_id,
                    "filename": seq_name,
                    "onset": onset_path,
                    "apex": out_apex_path,
                    "flow": out_flow_path,
                    "roi_paths": out_roi_paths,
                    "au_list": au_names,
                    "emotion": coarse_lower,
                    "coarse": coarse_lower,
                    "qa_list": qa_list
                })

    print(f"SMIC total seq : {n_total}")
    print(f"SMIC hit QA    : {n_hit_qa}")
    print(f"SMIC skip QA   : {n_skip_qa}")

    return data_infos


def build_samples(df, root_dir, dataset_type, out_dir):
    data_infos = []
    dataset_type = dataset_type.lower()

    au_extractor = AUExtractor("/kaggle/working/shape_predictor_68_face_landmarks.dat")

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Create sample [{dataset_type}]"):
        subject = int(row["Subject"])
        filename = str(row["Filename"]).strip()

        onset_frame = int(row["OnsetFrame"])
        apex_frame = int(row["ApexFrame"])

        au_raw = str(row.get("Action Units", ""))
        emotion = str(row.get("Estimated Emotion", "unknown")).lower()
        coarse = str(row.get("coarse emotion", "unknown")).lower()

        if dataset_type == "casme2":
            sub_folder = f"sub{subject:02d}"
            onset_path = os.path.join(root_dir, sub_folder, filename, f"reg_img{onset_frame}.jpg")
            apex_path = os.path.join(root_dir, sub_folder, filename, f"reg_img{apex_frame}.jpg")

            qa_candidates = [
                f"{sub_folder}_{filename}",
                filename,
                os.path.join(sub_folder, filename)
            ]
            qa_list = get_qa_list("casme2", *qa_candidates)
            subject_id = f"casme2_{sub_folder}"

        elif dataset_type == "samm":
            sub_folder = f"{subject:03d}"
            onset_path = os.path.join(root_dir, sub_folder, filename, f"crop_{onset_frame}.jpg")
            apex_path = os.path.join(root_dir, sub_folder, filename, f"crop_{apex_frame}.jpg")

            qa_candidates = [
                filename,
                f"{sub_folder}_{filename}",
                os.path.join(sub_folder, filename)
            ]
            qa_list = get_qa_list("samm", *qa_candidates)
            subject_id = f"samm_{sub_folder}"

        else:
            raise ValueError(f"Unsupported dataset_type: {dataset_type}")

        qa_list = get_qa_list(dataset_type, *qa_candidates)
        if len(qa_list) == 0:
            print(f"[Skip QA] {dataset_type} - {filename}")
            continue

        onset = cv2.imread(onset_path)
        apex = cv2.imread(apex_path)

        if onset is None or apex is None:
            print(f"[Skip IMG] Không đọc được ảnh: {onset_path} | {apex_path}")
            continue

        flow_img = get_optical_flow_image(onset, apex)

        landmarks = au_extractor.extract_landmarks(apex)
        if landmarks is None:
            print(f"[Skip FACE] Không detect được mặt: {dataset_type} - {filename}")
            continue

        au_names, roi_imgs = process_all_roi(apex, landmarks)

        if not roi_imgs or len(roi_imgs) == 0:
            print(f"[Skip ROI] {dataset_type} - {filename}")
            continue

        sample_id = os.path.join(subject_id, filename)
        sample_out_dir = os.path.join(out_dir, sample_id)
        os.makedirs(sample_out_dir, exist_ok=True)

        out_apex_path = os.path.join(sample_out_dir, "apex.jpg")
        out_flow_path = os.path.join(sample_out_dir, "flow.jpg")

        cv2.imwrite(out_apex_path, apex)
        cv2.imwrite(out_flow_path, flow_img)

        out_roi_paths = []
        for i, (name, roi_img) in enumerate(roi_imgs.items()):
            roi_path = os.path.join(sample_out_dir, f"roi_{i+1:02d}.jpg")
            cv2.imwrite(roi_path, roi_img)
            out_roi_paths.append(roi_path)

        data_infos.append({
            "dataset": dataset_type,
            "subject": subject_id,
            "filename": filename,
            "onset": onset_path,
            "apex": out_apex_path,
            "flow": out_flow_path,
            "roi_paths": out_roi_paths,
            "au_list": au_names,
            "emotion": emotion,
            "coarse": coarse,
            "qa_list": qa_list
        })

    return data_infos


# data_infos = build_samples(df, root_dir)
# print(f"Number of sample: {len(data_infos)}/{len(df)}")
# print("Done")

In [ ]:
# --- 1. CASME II ---
root_dir_casme = "/kaggle/input/datasets/tramanh1511tn/casme-dataset/Cropped/Cropped"
casme_infos = build_samples(
    df,
    root_dir=root_dir_casme,
    dataset_type="casme2",
    out_dir="megc_processed/casme2"
)

# --- 2. SAMM ---
root_dir_samm = "/kaggle/input/datasets/khuongvutramanh/samm-data-qq/samm-cropp/SAMM_Cropped"
samm_infos = build_samples(
    df_samm,
    root_dir=root_dir_samm,
    dataset_type="samm",
    out_dir="megc_processed/samm"
)

# --- 3. SMIC ---
root_dir_smic = "/kaggle/input/datasets/mnthbchphng/smic-dataset/SMIC_all_cropped/SMIC_all_cropped/HS"
smic_infos = build_smic_samples(
    root_dir=root_dir_smic,
    out_dir="megc_processed/smic"
)

# --- 4. GỘP 3 DATASET ---
total_data_infos = casme_infos + samm_infos + smic_infos
data_infos = total_data_infos

print(f"CASME2: {len(casme_infos)}")
print(f"SAMM  : {len(samm_infos)}")
print(f"SMIC  : {len(smic_infos)}")
print(f"TOTAL : {len(data_infos)}")

In [ ]:
from collections import Counter

print(Counter([x["dataset"] for x in data_infos]))

smic_examples = [x for x in data_infos if x["dataset"] == "smic"][:5]
for x in smic_examples:
    print(x["subject"], x["filename"], len(x["qa_list"]))
    for qa in x["qa_list"][:2]:
        print("  Q:", qa["question"])
        print("  A:", qa["answer"])

## TEXT PROMPT

In [ ]:
class PromptBuilder:
    def __init__(self):
        self.fine_emotions =ALL_EMOTION      # thiếu "repression" ở casme2, "contempt" ở samm
        self.coarse_emotions = ALL_COARSE

        # templates 
        self.au_templates = [
            "The {} is activated.",
            "The micro-expression shows the activation of the {}.",
            "A subtle facial movement activates the {}.",
            "The {} action unit is present on the face.",
            "A micro-expression involves the {}."
        ]
        
        self.fine_emo_templates = [
            "The micro-expression suggests {}.",
            "This is a micro-expression of {}.",
            "The expression indicates {}.",
            "The face briefly shows {}.",
            "A subtle expression of {} appears.",
            "This facial movement indicates {}."
        ]
        
        self.coarse_emo_templates = [
            "This micro-expression belongs to the {} emotion class.",
            "The emotion expressed is {}.",
            "The face shows a {} emotional state.",
            "This micro-expression is {}."
        ]
        
        # The action unit is lip corner puller, corresponding to happiness and positive emotion.
        # The lip corner puller is a key indicator happiness micro-expression, which is a positive state.
        self.joint_templates = [
            "The action units present is {}, which indicates a {} micro-expression, belonging to the {} emotion category.",
            "The facial action unit {} is a key indicator of {} micro-expression, which is a {} state.",
            "The presence of {} suggests a {} expression that belongs to the {} category."
        ]
    def format_au_list(self, au_list):
        if not au_list:
            return "no prominent action units"
        elif len(au_list) == 1:
            return au_list[0]
        elif len(au_list) == 2:
            return f"{au_list[0]} and {au_list[1]}"
        else:
            return ", ".join(au_list[:-1]) + f", and {au_list[-1]}"
            
    def get_au_prompt(self, au_names):
        if isinstance(au_names, str):
            au_names = [au_names]

        au_string = self.format_au_list(au_names)
        template = random.choice(self.au_templates)
        return template.format(au_string)

    def get_fine_emotion_prompt(self, fine_emotion):
        template = random.choice(self.fine_emo_templates)
        return template.format(fine_emotion)

    def get_coarse_emotion_prompt(self, coarse_emotion):
        template = random.choice(self.coarse_emo_templates)
        return template.format(coarse_emotion)

    def get_joint_prompt(self, au_names, fine_emotion, coarse_emotion):
        if isinstance(au_names, str):
            au_names = [au_names]

        au_string = self.format_au_list(au_names)
        
        template = random.choice(self.joint_templates)
        return template.format(
            au_string, 
            fine_emotion, 
            coarse_emotion
        )

    def build_sample_prompts(self, au_list, fine_emotion, coarse_emotion):
        return {
            "au_prompt": self.get_au_prompt(au_list),
            "fine_prompt": self.get_fine_emotion_prompt(fine_emotion),
            "coarse_prompt": self.get_coarse_emotion_prompt(coarse_emotion),
            "joint_prompt": self.get_joint_prompt(au_list,fine_emotion,coarse_emotion)
        }

prompt_builder = PromptBuilder()

In [ ]:
def test_prompt():
    
    # df = pd.read_excel("/kaggle/input/datasets/tramanh1511tn/casme-dataset/CASME2-coding-20140508.xlsx")
    # df = df[pd.to_numeric(df["ApexFrame"], errors="coerce").notna()]

    # lấy 1 mẫu visual test
    sample = df.iloc[8]
    print(sample)

    subject = sample["Subject"]
    filename = sample["Filename"]
    apex = int(sample["ApexFrame"])
    au_raw = str(sample["Action Units"])
    sub_folder = f"sub{subject:02d}"
    root_dir = "/kaggle/input/datasets/tramanh1511tn/casme-dataset/Cropped/Cropped"
    apex_path = os.path.join(root_dir, sub_folder, filename, f"reg_img{apex}.jpg")

    emotion = sample["Estimated Emotion"]
    # map từ số sang name
    au_names = get_name_au_label(au_raw)
    coarse = sample["coarse emotion"]
    

    
    print("\n--- AUs ---")
    print(prompt_builder.get_au_prompt(au_names))

    print("\n--- Emotion ---")
    print(prompt_builder.get_fine_emotion_prompt(emotion))

    print("\n--- Coarse ---")
    # thiếu hàm mapping từ emotion->coarse
    print(prompt_builder.get_coarse_emotion_prompt(coarse))
        
    print("\n--- JOINT PROMPT ---")
    # au_list_3 = ["cheek raiser", "lip corner puller", "dimpler"]
    print(prompt_builder.get_joint_prompt(
        au_names=au_names, 
        fine_emotion=emotion, 
        coarse_emotion="positive"
    ))

test_prompt()

# DATASET

In [ ]:
class MEDataset(Dataset):
    def __init__(self, data_infos, prompt_builder):
        self.data_infos = data_infos
        self.prompt_builder = prompt_builder

        
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225]),
        ])
        # self.transform = T.Compose([
        #     T.ToPILImage(),
        #     T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC),
        #     T.ColorJitter(brightness=0.2, contrast=0.2) if is_train else nn.Identity(),
        #     T.RandomRotation(degrees=5) if is_train else nn.Identity(), # Xoay nhẹ
        #     T.ToTensor(),
        #     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        # ])

    def __len__(self):
        return len(self.data_infos)

    def _load_image(self, path):
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Không tìm thấy ảnh tại: {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        info = self.data_infos[idx]
        
        apex_img = self._load_image(info['apex'])
        flow_img = self._load_image(info['flow'])
        roi_imgs = [self._load_image(p) for p in info['roi_paths']]

        apex_tensor = self.transform(apex_img)
        flow_tensor = self.transform(flow_img)
        roi_tensor = torch.stack([self.transform(roi) for roi in roi_imgs]) # [num_rois, 3, 224, 224]

        text_prompts = self.prompt_builder.build_sample_prompts(
            info['au_list'], 
            info['emotion'], 
            info['coarse']
        )
        
        return apex_tensor, flow_tensor, roi_tensor, text_prompts


In [ ]:
def collate_fn(batch):
    apex_list, flow_list, roi_list, text_list = zip(*batch)

    # stack apex & flow
    apex = torch.stack(apex_list)
    flow = torch.stack(flow_list)
    
    # ===== ROI padding =====
    max_rois = max(r.size(0) for r in roi_list)
    padded_rois = []
    for r in roi_list:
        pad_size = max_rois - r.shape[0]
        if pad_size > 0:
            pad = torch.zeros(pad_size, 3, 224, 224)
            r = torch.cat([r, pad], dim=0)
        padded_rois.append(r)

    roi = torch.stack(padded_rois)  # [B, max_roi, 3, 224, 224]


    # ===== text =====
    text_dict = {
        "au_prompts": [t["au_prompt"] for t in text_list],
        "fine_prompts": [t["fine_prompt"] for t in text_list],
        "coarse_prompts": [t["coarse_prompt"] for t in text_list],
        "joint_prompts": [t["joint_prompt"] for t in text_list]
    }

    return apex, flow, roi, text_dict

In [ ]:
def parse_vqa_question(question):
    q = question.lower().strip()

    if "left or right" in q:
        return "location", None
    
    yes_no_match = re.search(r"is the action unit (.*?) shown on the face", q)
    if yes_no_match:
        target_au = yes_no_match.group(1).strip()
        return "au_yes_no", target_au
        
    if "analyse" in q or "detailed" in q or "analysis" in q:
        return "joint", None
        
    if "coarse" in q:
        return "coarse", None
        
    if "fine-grained" in q or "fine" in q:
        return "fine", None
        
    if "action unit" in q:
        return "au", None
        
    return "unknown", None


In [ ]:
class MEVQA_Dataset(Dataset):
    def __init__(self, data_infos):
        # Mỗi file sẽ có số mẫu tương đương với 1 câu hỏi + 1 câu trả lời
        self.flat_data = []
        for info in data_infos:
            for qa in info.get('qa_list', []):
                self.flat_data.append({
                    "video_info": info,
                    "question": qa["question"],
                    "answer": qa["answer"]
                })
                
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.flat_data)

    def _load_image(self, path):
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Không tìm thấy ảnh tại: {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        item = self.flat_data[idx]
        info = item["video_info"]
        
        # 1. Load ảnh
        apex_img = self._load_image(info['apex'])
        flow_img = self._load_image(info['flow'])
        roi_imgs = [self._load_image(p) for p in info['roi_paths']]

        apex_tensor = self.transform(apex_img)
        flow_tensor = self.transform(flow_img)
        roi_tensor = torch.stack([self.transform(roi) for roi in roi_imgs]) 

        # 2. Load Câu hỏi & Trả lời
        question = item["question"]
        answer = item["answer"]
        
        # 3. phân loại Task cho mạng Router
        task_str, _ = parse_vqa_question(question)
        return apex_tensor, flow_tensor, roi_tensor, question, answer, task_str

In [ ]:
def vqa_collate_fn(batch):
    apex_list, flow_list, roi_list, q_list, a_list, task_list = zip(*batch)

    # 1. Stack ảnh tĩnh
    apex = torch.stack(apex_list)
    flow = torch.stack(flow_list)
    
    # 2. Padding ROI (Vì số lượng nếp nhăn mỗi video khác nhau)
    max_rois = max(r.size(0) for r in roi_list)
    padded_rois = []
    for r in roi_list:
        pad_size = max_rois - r.shape[0]
        if pad_size > 0:
            pad = torch.zeros(pad_size, 3, 224, 224)
            r = torch.cat([r, pad], dim=0)
        padded_rois.append(r)
    roi = torch.stack(padded_rois)  # [B, max_roi, 3, 224, 224]

    # 3. Convert chuỗi String thành List
    questions = list(q_list)
    answers = list(a_list)
    tasks = list(task_list)

    return apex, flow, roi, questions, answers, tasks

In [ ]:
subject_dict = defaultdict(list)
for item in data_infos:
    subject_dict[item["subject"]].append(item)

subjects = list(subject_dict.keys())
random.seed(42)
random.shuffle(subjects)

n = len(subjects)
# train_ratio=0.8
# val_ratio=0.1
train_ratio=0.9
n_train = int(n * train_ratio)
# n_val = int(n * val_ratio)

train_subjects = subjects[:n_train]
val_subjects = subjects[n_train:]

# val_subjects = subjects[n_train:n_train+n_val]
# test_subjects = subjects[n_train+n_val:]

def collect(sub_list):
    data = []
    for s in sub_list:
        data.extend(subject_dict[s])
    return data

train_data = collect(train_subjects)
val_data = collect(val_subjects)
# test_data = collect(test_subjects)

# print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print(f"Train: {len(train_data)}, Val: {len(val_data)}")



In [ ]:
# tạo data cho CLIP

train_dataset = MEDataset(train_data, prompt_builder)
val_dataset = MEDataset(val_data, prompt_builder)
# test_dataset = MEDataset(test_data, prompt_builder)

batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print("train", len(train_loader))
print("val", len(val_loader))
# print("test", len(test_loader))

In [ ]:
# # Thử in ra 1 batch để kiểm tra
# for apex, flow, roi, q_list, a_list, tasks in train_loader:
#     print("Batch size:", apex.shape[0])
#     print("Câu hỏi đầu tiên:", q_list[0])
#     print("Đáp án gốc:", a_list[0])
#     print("Mạng Router cần điều hướng vào Task:", tasks[0])
#     break

# VISION ENCODER

In [ ]:
class VisualEncoder(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", out_dim=512):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=True)
        self.vit.reset_classifier(0)

        # self.preprocess = transforms.Compose([
        #     transforms.ToPILImage(),
        #     transforms.Resize((224, 224)),
        #     transforms.ToTensor(),
        #     transforms.Normalize(mean=[0.485, 0.456, 0.406], 
        #                          std=[0.229, 0.224, 0.225]),
        # ])

        # self.proj = nn.Linear(self.vit.num_features, out_dim)
        self.proj = nn.Sequential(
                        nn.Linear(self.vit.num_features, out_dim),
                        nn.GELU(),
                        nn.Linear(out_dim, out_dim)
                    )

        
    def forward(self, x):
        feat = self.vit(x)
        out = self.proj(feat)
        return out


# vision_encoder = VisualEncoder()
# print(vision_encoder)

# TEXT ENCODER

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", device=None):
        super().__init__()
        
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device
            
        print(f"Loading CLIP model ({model_name}) on {self.device}...")
        self.model = CLIPModel.from_pretrained(model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_name)

        # freeze CLIP
        for param in self.model.parameters():
            param.requires_grad = False

    def forward(self, text_prompts):
        if isinstance(text_prompts, str):
            text_prompts = [text_prompts]
            
        # Tokenize văn bản
        inputs = self.processor(
            text=text_prompts, 
            return_tensors="pt", 
            padding=True,       
            truncation=True    
        ).to(self.device)
        
        # Trích xuất đặc trưng
        # text_features = self.model.get_text_features(**inputs)
        outputs = self.model.get_text_features(**inputs)
        text_features = outputs.pooler_output  # hoặc outputs.last_hidden_state        
        # Chuẩn hóa L2 (L2 Normalization) để tính Cosine Similarity hoặc làm đầu vào cho MoE
        # text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features

# CLIP ALignment

In [ ]:
class CLIPAlignment(nn.Module):
    def __init__(self, device, init_temp=0.07):
        super().__init__()
        self.device = device
        self.visual_encoder = VisualEncoder()
        self.text_encoder = TextEncoder()
        
        # Nhiệt độ InfoNCE
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / init_temp))

    def encode_visual(self, visual):
        visual_features = self.visual_encoder(visual)
        return F.normalize(visual_features, p=2, dim=-1)
    
    def encode_text(self, text_list):
        text_features = self.text_encoder(text_list)
        # text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return F.normalize(text_features, p=2, dim=-1)

    def compute_infonce(self, v_embeds, t_embeds):
        logit_scale = self.logit_scale.exp()
        logits = logit_scale * v_embeds @ t_embeds.t()
        labels = torch.arange(v_embeds.size(0), device=self.device)
        loss_v = F.cross_entropy(logits, labels)
        loss_t = F.cross_entropy(logits.t(), labels)
        return (loss_v + loss_t) / 2

    def forward(self, apex_imgs, flow_imgs, roi_imgs, text_prompts_dict):
        B = apex_imgs.size(0)
        
        # 1. TRÍCH XUẤT VISUAL FEATURES
        v_apex = self.visual_encoder(apex_imgs) # [B, 512]
        v_flow = self.visual_encoder(flow_imgs) # [B, 512]
        
        # Xử lý 14 ROIs: Flatten -> ViT -> Reshape -> Mean Pooling
        num_roi = roi_imgs.shape[1]
        v_rois_flat = self.visual_encoder(roi_imgs.view(-1, 3, 224, 224))
        v_roi = v_rois_flat.view(B, num_roi, -1).mean(dim=1) # [B, 512]
        v_roi = F.normalize(v_roi, p=2, dim=-1)

        # Tạo vector dung hợp (đã được L2 normalize)
        v_global = F.normalize(v_apex + v_flow, p=2, dim=-1)
        v_joint = F.normalize(v_apex + v_flow + v_roi, p=2, dim=-1)

        # 2. TRÍCH XUẤT TEXT FEATURES
        t_au = self.encode_text(text_prompts_dict['au_prompts'])
        t_fine = self.encode_text(text_prompts_dict['fine_prompts'])
        t_coarse = self.encode_text(text_prompts_dict['coarse_prompts'])
        t_joint = self.encode_text(text_prompts_dict['joint_prompts'])

        # 3. TÍNH 4 HÀM LOSS ĐỐI CHIẾU
        loss_au = self.compute_infonce(v_roi, t_au)
        loss_fine = self.compute_infonce(v_global, t_fine)
        loss_coarse = self.compute_infonce(v_global, t_coarse)
        loss_joint = self.compute_infonce(v_joint, t_joint)

        total_loss = (loss_au + loss_fine + loss_coarse + loss_joint) / 4.0
        
        return total_loss

    def predict_logits(self, apex_imgs, flow_imgs, roi_imgs, candidate_dict):
        self.eval() # Bật chế độ suy luận
        B = apex_imgs.size(0)
        
        with torch.no_grad():
            v_apex = self.visual_encoder(apex_imgs)
            v_flow = self.visual_encoder(flow_imgs)
            
            num_roi = roi_imgs.shape[1]
            v_rois_flat = self.visual_encoder(roi_imgs.view(-1, 3, 224, 224))
            v_roi = v_rois_flat.view(B, num_roi, -1).mean(dim=1)
            v_roi = F.normalize(v_roi, p=2, dim=-1)

            v_global = F.normalize(v_apex + v_flow, p=2, dim=-1)

            t_au = self.encode_text(candidate_dict['au']) 
            t_fine = self.encode_text(candidate_dict['fine']) 
            t_coarse = self.encode_text(candidate_dict['coarse']) 

            logit_scale = self.logit_scale.exp()
            
            logits_au = logit_scale * v_roi @ t_au.t()
            logits_fine = logit_scale * v_global @ t_fine.t()
            logits_coarse = logit_scale * v_global @ t_coarse.t()
            
        return logits_au, logits_fine, logits_coarse

# MOE ARCHITECTURE

In [ ]:
# class QuestionRouter(nn.Module):
#     def __init__(self, embed_dim=512, num_experts=4):
#         super().__init__()
#         self.gate = nn.Sequential(
#             nn.Linear(embed_dim, 128),
#             nn.Tanh(),
#             nn.Linear(128, num_experts)
#         )
#     def forward(self, q_text):
#         logits = self.gate(q_text)
#         return F.softmax(logits, dim=-1)


# class ExpertAULocal(nn.Module):
#     def __init__(self, embed_dim=512, num_heads=8):
#         super().__init__()
#         self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
#         self.layer_norm = nn.LayerNorm(embed_dim)
#         self.ffn = nn.Sequential(
#             nn.Linear(embed_dim, embed_dim),
#             nn.GELU(),
#             nn.Dropout(0.1)
#         )

#     def forward(self, v_roi, q_text):
#         q = q_text.unsqueeze(1) 
#         attn_out, _ = self.cross_attn(query=q, key=v_roi, value=v_roi)
#         out = self.layer_norm(q + attn_out).squeeze(1) # [B, 512]
#         return self.ffn(out)


class ExpertAULocal(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8):
        super().__init__()
        # 1. Lớp Self-Attention: Giúp 17 vùng ROI "nói chuyện" với nhau
        self.roi_self_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        
        # 2. Lớp Cross-Attention: Câu hỏi rà soát các ROI đã được làm giàu thông tin
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim * 2, embed_dim)
        )

    def forward(self, v_roi, q_text):
        roi_context, _ = self.roi_self_attn(v_roi, v_roi, v_roi)
        v_roi = self.norm1(v_roi + roi_context)
        
        # Bước 2: Question-guided spotting
        q = q_text.unsqueeze(1) 
        attn_out, _ = self.cross_attn(query=q, key=v_roi, value=v_roi)
        out = self.norm2(q + attn_out).squeeze(1)
        
        return self.ffn(out)

# class ExpertEmotionHolistic(nn.Module):
#     def __init__(self, embed_dim=512):
#         super().__init__()
#         self.fusion = nn.Sequential(
#             nn.Linear(embed_dim * 2, embed_dim),
#             nn.LayerNorm(embed_dim),
#             nn.GELU(),
#             nn.Linear(embed_dim, embed_dim)
#         )

#     def forward(self, v_global, q_text):
#         fused_input = torch.cat([v_global, q_text], dim=-1) 
#         out = self.fusion(fused_input) 
#         return out

class ExpertEmotionHolistic(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8):
        super().__init__()
        # Dùng Transformer để hòa trộn (Fusion) thay vì MLP
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            batch_first=True,
            dim_feedforward=embed_dim * 2,
            activation='gelu',
            dropout=0.1
        )
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        
        # Token đặc biệt để gom thông tin (Giống [CLS] token trong BERT)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, v_global, q_text):
        B = v_global.size(0)
        
        # Mở rộng chiều sequence
        v = v_global.unsqueeze(1) # [B, 1, 512]
        q = q_text.unsqueeze(1)   # [B, 1, 512]
        cls = self.cls_token.expand(B, -1, -1) # [B, 1, 512]
        
        tokens = torch.cat([cls, v, q], dim=1) # [B, 3, 512]
        
        # Cho qua Transformer
        fused_tokens = self.fusion_transformer(tokens)        
        out = fused_tokens[:, 0, :] # [B, 512]
        return out

class ExpertSpatial(nn.Module):
    def __init__(self, embed_dim=512, num_rois=17, num_heads=8):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, num_rois, embed_dim) * 0.02)
        
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU()
        )

    def forward(self, v_roi, q_text):
        v_roi_spatial = v_roi + self.pos_embed
        
        q = q_text.unsqueeze(1)
        attn_out, _ = self.cross_attn(query=q, key=v_roi_spatial, value=v_roi_spatial)
        
        out = self.layer_norm(q + attn_out).squeeze(1)
        return self.ffn(out)

class ExpertRelation(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            batch_first=True,
            dim_feedforward=embed_dim * 2,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, out_au, out_emo, q_text):
        # Lấy output từ 2 chuyên gia au và emotion
        # Xếp 3 vector thành một chuỗi (Sequence) có độ dài = 3
        nodes = torch.stack([out_au, out_emo, q_text], dim=1)
        reasoned_nodes = self.transformer_encoder(nodes)
        out = reasoned_nodes.mean(dim=1) # [Batch, 512]
        return out


In [ ]:
class HierarchicalMoE(nn.Module):
    def __init__(self, device, pretrain_visual_path, embed_dim=512, init_temp=0.07):
        super().__init__()
        self.device = device

        self.visual_encoder = torch.load(pretrain_visual_path, map_location=device, weights_only=False)
        self.text_encoder = TextEncoder(device=device)
        
        print(f"Loading pre-trained Visual Encoder from {pretrain_visual_path} and freeze... ")
        for param in self.visual_encoder.parameters():
            param.requires_grad = False

        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / init_temp))
        
        self.router = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 6) # 5 Tasks: AU, AU_Loc, Coarse, Fine, Joint --> chuyển thành 6 task thêm 1 task cho location (left, right)???
        )
        
        # Learnable Matrix ánh xạ từ 6 Tasks sang 4 Experts -> tự phân chia weight của expert
        self.task_to_expert_proj = nn.Linear(6, 4) 
        
        self.e_au_local = ExpertAULocal(embed_dim)
        self.e_emo_holistic = ExpertEmotionHolistic(embed_dim)
        self.e_spatial = ExpertSpatial(embed_dim)
        self.e_relation = ExpertRelation(embed_dim)
        
        self.final_norm = nn.LayerNorm(embed_dim)

    # def forward(self, v_apex, v_flow, v_roi, q_text):
    def forward(self, apex_imgs, flow_imgs, roi_imgs, questions_list, target_answers=None):
        """
        v_roi: [Batch, 14, 512]
        v_global (apex+flow): [Batch, 512]
        q_text: [Batch, 512]
        """
        B = apex_imgs.size(0)
        
        # --- BƯỚC 1: TRÍCH XUẤT ĐẶC TRƯNG TĨNH (No Grad) ---
        with torch.no_grad():
            v_apex = self.visual_encoder(apex_imgs)
            v_flow = self.visual_encoder(flow_imgs)
            
            num_roi = roi_imgs.shape[1]
            v_rois_flat = self.visual_encoder(roi_imgs.view(-1, 3, 224, 224))
            v_roi = v_rois_flat.view(B, num_roi, -1)#.mean(dim=1)

            v_global = F.normalize(v_apex + v_flow, p=2, dim=-1)
            
            q_text = self.text_encoder(questions_list) # [B, 512]
        
        task_logits = self.router(q_text)
        task_probs = F.softmax(task_logits, dim=-1)
        
        # expert_weights: [Batch, 4] chứa tỷ trọng phân bổ cho 4 chuyên gia
        expert_logits = self.task_to_expert_proj(task_probs)
        expert_weights = F.softmax(expert_logits, dim=-1)
        
        w_au, w_emo, w_spa, w_rel = torch.split(expert_weights, 1, dim=-1)

        # STAGE 2: CÁC CHUYÊN GIA LÀM VIỆC SONG SONG
        out_au  = self.e_au_local(v_roi, q_text)
        out_emo = self.e_emo_holistic(v_global, q_text)
        out_spa = self.e_spatial(v_roi, q_text)
        
        out_rel = self.e_relation(out_au, out_emo, q_text)
        
        # 3. Router cấp quyền (Expert Fusion)
        # (w_au, w_emo, w_spa, w_rel là tỷ lệ do Stage 1 quyết định
        v_final = (w_au * out_au) + (w_emo * out_emo) + (w_spa * out_spa) + (w_rel * out_rel)                  
        v_final = self.final_norm(v_final)

        v_final = F.normalize(v_final, p=2, dim=-1)

        # --- BƯỚC 5: TÍNH LOGITS CHO QUÁ TRÌNH TRAIN ---
        if target_answers is not None:
            t_answers = self.text_encoder(target_answers)
            t_answers = F.normalize(t_answers, p=2, dim=-1)
            
            logit_scale = self.logit_scale.exp()
            logits_vqa = logit_scale * v_final @ t_answers.t() # [B, B]
            return logits_vqa, task_logits, expert_weights
            
        return v_final, task_logits, expert_weights
        
    def predict(self, apex_imgs, flow_imgs, roi_imgs, question, candidate_texts):
        self.eval()
        with torch.no_grad():
            v_fused, _, _ = self.forward(apex_imgs, flow_imgs, roi_imgs, [question]*apex_imgs.size(0))
            
            t_candidates = self.text_encoder(candidate_texts)
            t_candidates = F.normalize(t_candidates, p=2, dim=-1)
            
            logit_scale = self.logit_scale.exp()
            logits = logit_scale * v_fused @ t_candidates.t() # [Batch, Num_Candidates]
            
        return logits

# OPTIMIZER

**CLIP ALIGNMENT**

In [ ]:
# # optim for CLIP

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = CLIPAlignment(device=device)
# trainable_params = [p for p in model.parameters() if p.requires_grad]

# epochs = 20
# optimizer = optim.AdamW(trainable_params, lr=5e-5, weight_decay=1e-4)
# scheduler = CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

In [ ]:
# TRAIN CLIP

def train_model(model, train_loader, val_loader, epochs, device="cuda"):
    model = model.to(device)
    
    best_val_loss = float('inf')
    save_dir = "checkpoints"
    os.makedirs(save_dir, exist_ok=True)

    train_loss_history = list()
    val_loss_history = list()

        
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        # for apex, flow, roi, texts in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]"):
        for apex, flow, roi, texts in train_loader:
            apex, flow, roi = apex.to(device), flow.to(device), roi.to(device)
            
            optimizer.zero_grad()
            loss = model(apex, flow, roi, texts)
            
            # Backward & Cập nhật trọng số
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            # print(f"Train loss f{loss.item():.4f}")
            
        avg_train_loss = train_loss / len(train_loader)
        train_loss_history.append(avg_train_loss)
        scheduler.step() # Giảm Learning Rate


        # =========VALIDATION============
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad(): 
            # for apex, flow, roi, texts in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [VAL]"):
            for apex, flow, roi, texts in val_loader:
                apex, flow, roi = apex.to(device), flow.to(device), roi.to(device)
                
                loss = model(apex, flow, roi, texts)
                val_loss += loss.item()
                                
        avg_val_loss = val_loss / len(val_loader)
        val_loss_history.append(avg_val_loss)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | Current learning rate = {current_lr:.8f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.visual_encoder, os.path.join(save_dir, "best_visual_encoder.pth"))
            print(f"*** Save model best: (Val Loss: {best_val_loss:.4f}) ***\n")
        else:
            print("\n")

    
    # =============== PLOT TRAINING CURVES ===============
    f, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    f.suptitle('Training Results')
    ax1.plot(range(0, len(train_loss_history)), train_loss_history, color='Blue', label='Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(range(0, len(val_loss_history)), val_loss_history, color='Red', label='Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Validation Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()

            
    print("compelte training")

In [ ]:
# print(f"Training with {epochs} epochs")
# train_model(model, train_loader, val_loader, epochs, device)

In [ ]:
import gc, torch

for name in ["model", "optimizer", "scheduler", "train_loader", "val_loader", "train_dataset", "val_dataset"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary(device=None, abbreviated=True))

**MOE ARCHITECTURE**

In [ ]:
def parse_vqa_question(question):
    q = question.lower().strip()

    if "left or right" in q:
        return "location", None
    
    yes_no_match = re.search(r"is the action unit (.*?) shown on the face", q)
    if yes_no_match:
        target_au = yes_no_match.group(1).strip()
        return "au_yes_no", target_au
        
    if "analyse" in q or "detailed" in q or "analysis" in q:
        return "joint", None
        
    if "coarse" in q:
        return "coarse", None
        
    if "fine-grained" in q or "fine" in q:
        return "fine", None
        
    # if "action unit" in q:
    #     return "au", None

    if "what are the action units" in q or "action units present" in q:
        return "au_plural", None
        
    if "action unit" in q:
        return "au_singular", None
        
    return "unknown", None


In [ ]:
# TRAIN MOE

TASK_MAP = {
    # "au": 0,
    "au_singular": 0, # giao cho au expert
    "au_plural": 0,   # giao cho au expert
    "au_yes_no": 1,
    "coarse": 2,
    "fine": 3,
    "joint": 4,
    "location": 5
}

def compute_multi_task_loss(logits_vqa, task_logits, true_task_ids, device):
    # 1. VQA InfoNCE Loss (In-batch negatives)
    labels_vqa = torch.arange(logits_vqa.size(0)).to(device)
    loss_v_to_t = F.cross_entropy(logits_vqa, labels_vqa)
    loss_t_to_v = F.cross_entropy(logits_vqa.t(), labels_vqa)
    loss_vqa = (loss_v_to_t + loss_t_to_v) / 2.0
    
    # 2. Router Guidance Loss
    loss_router = F.cross_entropy(task_logits, true_task_ids)
    
    # Cộng gộp (Hệ số 0.5 giúp cân bằng gradient)
    return loss_vqa + 0.5 * loss_router

def train_moe_model(model, train_loader, val_loader, epochs, device="cuda"):
    model = model.to(device)
        
    best_loss = float('inf')
    save_dir = "checkpoints_moe"
    os.makedirs(save_dir, exist_ok=True)

    train_loss_history = list()
    val_loss_history = list()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for apex, flow, roi, q_list, a_list, task_strs in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]"):
            apex, flow, roi = apex.to(device), flow.to(device), roi.to(device)            
            true_task_ids = torch.tensor([TASK_MAP.get(t, 0) for t in task_strs]).to(device)
            
            optimizer.zero_grad()
            
            logits_vqa, task_logits, expert_weights = model(apex, flow, roi, q_list, target_answers=a_list)
            
            # Loss & Backward
            loss = compute_multi_task_loss(logits_vqa, task_logits, true_task_ids, device)
            loss.backward()
            
            # Chống nổ gradient cho Transformer/Attention
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        train_loss_history.append(avg_train_loss)
        scheduler.step()
        
        # ================= VALIDATE =================
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for apex, flow, roi, q_list, a_list, task_strs in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [VAL]"):
                apex, flow, roi = apex.to(device), flow.to(device), roi.to(device)
                true_task_ids = torch.tensor([TASK_MAP.get(t, 0) for t in task_strs]).to(device)
                
                logits_vqa, task_logits, _ = model(apex, flow, roi, q_list, target_answers=a_list)
                loss = compute_multi_task_loss(logits_vqa, task_logits, true_task_ids, device)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_loss_history.append(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.7f}")
        
        # Save best model
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            # Lưu state_dict cho an toàn và nhẹ
            torch.save(model, os.path.join(save_dir, "best_moe_megc.pth"))
            print(f"*** Đã lưu mô hình MoE tốt nhất: (Loss: {best_loss:.4f}) ***\n")
        else:
            print("\n")

    
    # =============== PLOT TRAINING CURVES ===============
    f, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    f.suptitle('Training Results')
    ax1.plot(range(0, len(train_loss_history)), train_loss_history, color='Blue', label='Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(range(0, len(val_loss_history)), val_loss_history, color='Red', label='Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Validation Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print("Complete training")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10

pretrained_clip_path = "/kaggle/input/datasets/ngtht71/train-clip/best_visual_encoder.pth"
model = HierarchicalMoE(device=device, pretrain_visual_path=pretrained_clip_path)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=3e-5, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

In [ ]:
# tạo data cho MoE
train_dataset = MEVQA_Dataset(train_data)
val_dataset = MEVQA_Dataset(val_data)
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=vqa_collate_fn)


batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=vqa_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, collate_fn=vqa_collate_fn)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print("train", len(train_loader))
print("val", len(val_loader))

In [ ]:
print(f"Training with {epochs} epochs")
train_moe_model(model, train_loader, val_loader, epochs, device)

# INFERENCE

In [ ]:
import dlib
import cv2
import numpy as np


def extract_face(image):
    datFile = "/kaggle/working/shape_predictor_68_face_landmarks.dat"
    face_detector = dlib.get_frontal_face_detector()
    landmark_predictor = dlib.shape_predictor(datFile)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_detector(gray)
    if len(faces) == 0:
        return None
    
    face = faces[0]
    landmarks = landmark_predictor(gray, face)
    points = np.array([(p.x, p.y) for p in landmarks.parts()])

    # === Lấy vùng mặt (giữ trán, bỏ tai) ===
    x_left = points[2][0]       # má trái
    x_right = points[14][0]     # má phải

    # Tính điểm cao nhất trên trán giữa hai lông mày (giữ trán)
    y_top = max(0, min(points[19][1], points[24][1]) - 60)
    y_bottom = points[8][1]

    # Cắt ảnh khuôn mặt
    face_img = image[y_top:y_bottom, x_left:x_right]
    if face_img.size == 0:
        return None

    face_resized = cv2.resize(face_img, (224, 224))
    return face_resized


def process_crop_image():
    base_path = '/kaggle/input/unseen-data-megc2026/ME_VQA_MEGC_2025_Test'
    save_base = '/kaggle/working/ME_VQA_MEGC_2025_Test'
    os.makedirs(save_base, exist_ok=True)
    
    rows = []
    video_folders = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))])
    
    for folder_name in tqdm(video_folders, desc="Processing Videos"):
        video_path = os.path.join(base_path, folder_name)
        
        # Lấy toàn bộ file ảnh trong thư mục này
        valid_exts = ('.jpg', '.jpeg', '.png')
        all_files = sorted([f for f in os.listdir(video_path) if f.lower().endswith(valid_exts)])
        
        if len(all_files) < 2: # Ít nhất phải có 2 ảnh để làm onset/apex
            continue
    
        # Xác định Onset và Apex
        onset_filename = all_files[0]
        apex_filename = all_files[len(all_files) // 2]
        
        # Tạo thư mục lưu cho video này
        save_video_dir = os.path.join(save_base, folder_name)
        os.makedirs(save_video_dir, exist_ok=True)
    
        # Biến đánh dấu nếu crop thành công cả 2 ảnh
        success = True
        
        for type_name, file_name in [("onset", onset_filename), ("apex", apex_filename)]:
            img_path = os.path.join(video_path, file_name)
            image = cv2.imread(img_path)
            
            if image is not None:
                face = extract_face(image) 
                if face is not None:
                    cv2.imwrite(os.path.join(save_video_dir, f"{type_name}.jpg"), face)
                else:
                    success = False
            else:
                success = False
        # Nếu cả onset và apex đều crop ổn, lưu vào data
        if success:
            rows.append({
                "video": folder_name,
                "onset_frame": os.path.splitext(onset_filename)[0],
                "apex_frame": os.path.splitext(apex_filename)[0],
                # "path": save_video_dir
            })
    
        else:
            print(f"Không tìm thấy khuôn mặt trong video {folder_name}")

process_crop_image()

# đọc jsonl
data = []
with open("/kaggle/working/me_vqa_samm_v2_test_to_answer.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))
with open("/kaggle/working/me_vqa_casme3_v2_test_to_answer.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

json_df = pd.DataFrame(data)
json_df

# coarse_df = json_df[
#     (json_df["dataset"] == "casme2") &
#     (json_df["question"] == "What is the coarse expression class?")
# ][["filename", "subject", "answer"]]
# coarse_df = coarse_df.rename(columns={"answer": "coarse emotion", "filename":"Filename", "subject":"Subject"})
# coarse_df["Subject"] = coarse_df["Subject"].str.replace("sub", "").astype(int)


df = json_df.drop_duplicates(subset='video')
df = df.reset_index(drop=True)


root_dir = "/kaggle/working/ME_VQA_MEGC_2025_Test"
out_dir = "test_processed"

def build_samples_inference(df, root_dir):
    data_infos = []
    count = 0

    for idx, row in df.iterrows():
        if count % 50 == 0:
            print(f"Proceesed {count}/{len(df)}")
        count +=1
        
        
        # subject = int(row["Subject"])
        # filename = row["Filename"]
        
        # onset_frame = int(row["OnsetFrame"])
        # apex_frame = int(row["ApexFrame"])
        
        # au_raw = str(row["Action Units"])
        # emotion = row["Estimated Emotion"]
        # coarse = row["coarse emotion"]
        video = row["video"]

        onset_path = os.path.join(root_dir, video, f"onset.jpg")
        apex_path = os.path.join(root_dir, video, f"apex.jpg")

        # lấy ảnh optical flow
        onset = cv2.imread(onset_path)
        apex = cv2.imread(apex_path)
        flow_img = get_optical_flow_image(onset, apex)

        # lấy ảnh roi cho tất cả vùng đã gán nhãn
        au_extractor = AUExtractor("/kaggle/working/shape_predictor_68_face_landmarks.dat")
        landmarks = au_extractor.extract_landmarks(apex)

        if landmarks is None:
            print(f"Skip image: no face detected in image {video}")
            continue
        au_names, roi_imgs = process_all_roi(apex, landmarks)

        # lưu output
        # sample_id = os.path.join(sub_folder, filename)
        sample_out_dir = os.path.join(out_dir, video)
        os.makedirs(sample_out_dir, exist_ok=True)

        out_apex_path = os.path.join(sample_out_dir, "apex.jpg")
        out_flow_path = os.path.join(sample_out_dir, "flow.jpg")

        # lưu các ảnh
        cv2.imwrite(out_apex_path, apex)
        cv2.imwrite(out_flow_path, flow_img)        
        out_roi_paths = []
        for i, (name, roi_img) in enumerate(roi_imgs.items()):
            ind = i+1
            roi_path = os.path.join(sample_out_dir, f"roi_{ind:02d}.jpg")
            cv2.imwrite(roi_path, roi_img)
            out_roi_paths.append(roi_path)

        data_infos.append({
            "video": video,
            "onset": onset_path,
            "apex": out_apex_path,
            "flow": out_flow_path,
            'roi_paths': out_roi_paths,
        })


    return data_infos
data_infos = build_samples_inference(df, root_dir)
print(f"Number of sample: {len(data_infos)}/{len(df)}")
print("Done")

In [ ]:

candidate_dict = {
    "au": ALL_AU_NAME,
    "fine": ALL_EMOTION,
    "coarse": ALL_COARSE,
    "location": ["left", "right"],
    "au_yes_no": ["yes", "no"]
}

def generate_answer(model, apex, flow, roi, question, device):
    model.eval()
    task, target_au = parse_vqa_question(question)

    if task == "unknown":
        return "Sorry, I cannot understand the question format."

    # all_aus = ALL_AU_NAME
    # all_fines = ALL_EMOTION
    # all_coarses = ALL_COARSE
        
    # Đảm bảo tensor có chiều Batch [1, ...]
    if apex.dim() == 3: apex = apex.unsqueeze(0)
    if flow.dim() == 3: flow = flow.unsqueeze(0)
    if roi.dim() == 4: roi = roi.unsqueeze(0)
        
    apex, flow, roi = apex.to(device), flow.to(device), roi.to(device)


    with torch.no_grad():
        # dự đoán nhiều AU
        if task == "au_plural":
            candidate_texts = candidate_dict.get("au", [])
            logits = model.predict(apex, flow, roi, question, candidate_texts)
            
            # Chuyển Logits thành Xác suất (Probabilities)
            probs = torch.softmax(logits, dim=1)[0] 
            
            max_prob = probs.max().item()
            threshold = max_prob * 0.6            # ngưỡng là cao hơn 0.55 cái cao nhất
            
            selected_indices = torch.where(probs >= threshold)[0].tolist()
            
            predicted_aus = [candidate_texts[i] for i in selected_indices]
            answer = ", ".join(predicted_aus)
            return answer

        # dự đoán 1 AU
        elif task == "au_singular":
            candidate_texts = candidate_dict.get("au", [])
            logits = model.predict(apex, flow, roi, question, candidate_texts)
            
            pred_idx = torch.argmax(logits, dim=1).item()
            return candidate_texts[pred_idx]

        elif task == "joint":
            # 1. Dự đoán AU
            logits_au = model.predict(apex, flow, roi, question, candidate_dict["au"])
            pred_au = candidate_dict["au"][torch.argmax(logits_au, dim=1).item()]
            
            # 2. Dự đoán Fine-grained Emotion
            logits_fine = model.predict(apex, flow, roi, question, candidate_dict["fine"])
            pred_fine = candidate_dict["fine"][torch.argmax(logits_fine, dim=1).item()]
            
            # 3. Dự đoán Coarse Emotion
            logits_coarse = model.predict(apex, flow, roi, question, candidate_dict["coarse"])
            pred_coarse = candidate_dict["coarse"][torch.argmax(logits_coarse, dim=1).item()]
            
            answer = f"The observed facial movement corresponds to action unit {pred_au}, which is associated with {pred_fine} and an overall {pred_coarse} emotion."
            return answer

        else:
            candidate_texts = candidate_dict.get(task, [])
            if not candidate_texts:
                return "unknown"
                
            logits = model.predict(apex, flow, roi, question, candidate_texts)
            pred_idx = torch.argmax(logits, dim=1).item()
            return candidate_texts[pred_idx]

        # if task == "joint":
            # # 1. Dự đoán AU
            # logits_au = model.predict(apex, flow, roi, question, candidate_dict["au"])
            # pred_au = candidate_dict["au"][torch.argmax(logits_au, dim=1).item()]
            
            # # 2. Dự đoán Fine-grained Emotion
            # logits_fine = model.predict(apex, flow, roi, question, candidate_dict["fine"])
            # pred_fine = candidate_dict["fine"][torch.argmax(logits_fine, dim=1).item()]
            
            # # 3. Dự đoán Coarse Emotion
            # logits_coarse = model.predict(apex, flow, roi, question, candidate_dict["coarse"])
            # pred_coarse = candidate_dict["coarse"][torch.argmax(logits_coarse, dim=1).item()]
            
            # answer = f"The observed facial movement corresponds to action unit {pred_au}, which is associated with {pred_fine} and an overall {pred_coarse} emotion."
            # return answer
            
        # else:
        #     # Lấy danh sách đáp án tương ứng với Task (vd: ["yes", "no"] hoặc ["left", "right"])
        #     candidate_texts = candidate_dict.get(task, [])            
        #     logits = model.predict(apex, flow, roi, question, candidate_texts)
            
        #     # Lấy đáp án có điểm cao nhất
        #     pred_idx = torch.argmax(logits, dim=1).item()
        #     answer = candidate_texts[pred_idx]
            
        #     return answer
    
    

def load_and_transform_video(info):
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    def _load(path):
        img = cv2.imread(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    apex = transform(_load(info['apex']))
    flow = transform(_load(info['flow']))
    roi = torch.stack([transform(_load(p)) for p in info['roi_paths']])
    
    return apex, flow, roi

def process_submission_file(input_jsonl, output_jsonl, model, device, data_infos):
    test_dataset = MEVQA_Dataset(data_infos)
    
    video_to_info = {info["video"]: info for info in data_infos}

    video_to_tasks = defaultdict(list)
    with open(input_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                task = json.loads(line)
                video_name = task["video"]
                video_to_tasks[video_name].append(task)
                
    print(f"Đã đọc {sum(len(v) for v in video_to_tasks.values())} câu hỏi từ {len(video_to_tasks)} videos.")
    
    # 4. XỬ LÝ TỪNG VIDEO
    results = []
    for video_name, tasks in tqdm(video_to_tasks.items(), desc="Processing Videos"):
        try:
            if video_name not in video_to_info:
                print(f"\n[Cảnh báo] Video '{video_name}' không có trong test dataset. Bỏ qua!")
                results.extend(tasks)
                continue
                
            # LẤY INDEX VÀ GỌI TRỰC TIẾP TỪ DATASET
            info = video_to_info[video_name]
            apex, flow, roi = load_and_transform_video(info)

            # idx = video_to_idx[video_name]
            # apex, flow, roi = test_dataset[idx] # Hàm __getitem__ sẽ tự động đọc ảnh và transform!
            
            # apex = apex.to(device)
            # flow = flow.to(device)
            # roi = roi.to(device)
            
            # Giải quyết mọi câu hỏi của video này
            for task in tasks:
                question = task["question"]
                answer = generate_answer(model, apex, flow, roi, question, device)
                task["answer"] = answer
                results.append(task)
                
        except Exception as e:
            print(f"\n[Lỗi] Không thể xử lý video {video_name}: {str(e)}")
            for task in tasks:
                results.append(task)

                
    # 5. GHI KẾT QUẢ
    print(f"\nĐang ghi kết quả ra file: {output_jsonl}")
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for res in results:
            f.write(json.dumps(res, ensure_ascii=False) + '\n')
            
    print("Done")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = HierarchicalMoE()

# save_dir = "checkpoints"
weights_path = "/kaggle/working/checkpoints_moe/best_moe_megc.pth"

if os.path.exists(weights_path):
    print(f"Đang nạp trọng số từ {weights_path}...")
    model = torch.load(weights_path, map_location=device, weights_only=False)
else:
    print("Cảnh báo: Không tìm thấy trọng số.")

In [ ]:
input_jsonl = "/kaggle/working/me_vqa_samm_v2_test_to_answer.jsonl"
output_jsonl = "/kaggle/working/me_vqa_samm_v2_test_pred.jsonl"
process_submission_file(input_jsonl, output_jsonl, model, device, data_infos)

In [ ]:
input_jsonl = "/kaggle/working/me_vqa_casme3_v2_test_to_answer.jsonl"
output_jsonl = "/kaggle/working/me_vqa_casme3_v2_test_pred.jsonl"
process_submission_file(input_jsonl, output_jsonl, model, device, data_infos)

In [ ]:
!zip -j me_vqa_v2_test_pred.zip \
    "/kaggle/working/me_vqa_samm_v2_test_pred.jsonl" \
    "/kaggle/working/me_vqa_casme3_v2_test_pred.jsonl"

In [ ]:
# # 2. Định nghĩa danh sách các câu hỏi test
# test_questions = [
#     "What is the coarse expression class?",
#     "What is the fine-grained expression class?",
#     "What is the action unit?",
#     "Is the action unit lip corner puller shown on the face?",
#     "Is the action unit lower lip depressor shown on the face?",
#     "Please analyse the micro-expression shown in this video in detail."
# ]

# # 3. Trích xuất thử với 1 Sample (Ví dụ bạn lấy video đầu tiên từ Dataloader)
# apex, flow, roi, _ = next(iter(test_loader))
# # Lấy sample số 0: apex[0], flow[0], roi[0]

# print("=== KẾT QUẢ VQA ZERO-SHOT ===")
# for q in test_questions:
#     # Truyền sample vào engine để lấy đáp án
#     ans = generate_answer(model, apex[0], flow[0], roi[0], q, device)
    
#     # Định dạng output chuẩn JSON-like string
#     output_line = f'"question": "{q}", "answer": "{ans}"'
#     print(output_line)


In [ ]:
# def inference_vqa(model, test_loader, prompt_builder, question, device):
#     model.eval()
#     task = classify_question(question)

#     # ===== build candidate prompts =====
#     # candidate_dict, ALL_AU_NAME, ALL_EMOTION, ALL_COARSE = build_candidate_prompts(prompt_builder)

    
#     all_preds_idx = []
#     all_labels_text = []

#     # Xác định danh sách Class đang xét để tính Metrics
#     if task == "au":
#         class_names = ALL_AU_NAME
#         target_dict_key = "au_prompts"
#     elif task == "coarse":
#         class_names = ALL_COARSE
#         target_dict_key = "coarse_prompts"
#     else:
#         # Nếu là câu hỏi Joint hoặc Fine, ta mặc định chấm điểm cho Fine Emotion
#         task = "fine" 
#         class_names = ALL_EMOTION
#         target_dict_key = "fine_prompts"
    
#     with torch.no_grad():
#         for apex, flow, roi, text_dict in tqdm(test_loader, desc=f"Inference [{task.upper()}]"):
#             apex, flow, roi = apex.to(device), flow.to(device), roi.to(device)

#             # Tính Logits
#             logits_au, logits_fine, logits_coarse = model.predict_logits(apex, flow, roi, candidate_dict)

#             # Chọn Logits dựa trên Câu hỏi
#             if task == "au":
#                 preds = torch.argmax(logits_au, dim=1).cpu().numpy()
#             elif task == "coarse":
#                 preds = torch.argmax(logits_coarse, dim=1).cpu().numpy()
#             else:
#                 preds = torch.argmax(logits_fine, dim=1).cpu().numpy()

#             # Lấy Ground Truth thô từ DataLoader
#             gt_texts = text_dict[target_dict_key]

#             # Lưu lại để tính toán
#             all_preds_idx.extend(preds)
#             all_labels_text.extend(gt_texts)

#     return all_preds_idx, all_labels_text, class_names

# # Tìm index của class trong câu prompt Ground Truth.
# def convert_labels_to_index(labels_text, class_list):
#     indices = []
#     for text in labels_text:
#         matched_idx = -1
#         for idx, name in enumerate(class_list):
#             if name.lower() in text.lower():
#                 matched_idx = idx
#                 break
        
#         # Fallback an toàn nếu chuỗi text bị lỗi
#         if matched_idx == -1:
#             matched_idx = 0 
#         indices.append(matched_idx)
#     return indices

# def evaluate_megc(model, test_loader, prompt_builder, question, device):
#     # 1. Chạy Inference lấy danh sách Dự đoán & Nhãn thực
#     preds_idx, labels_text, class_names = inference_vqa(model, test_loader, prompt_builder, question, device)

#     # 2. Mapping chuỗi Text Ground Truth về dạng Số (Index)
#     y_true = convert_labels_to_index(labels_text, class_names)
#     y_pred = preds_idx

#     # 3. Tính toán Metrics
#     num_classes = len(class_names)
#     f1_per_class = f1_score(
#         y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
#     )
#     recall_per_class = recall_score(
#         y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
#     )

#     UF1 = f1_per_class.mean()
#     UAR = recall_per_class.mean()

#     # 4. In Báo cáo
#     print("\n" + "="*40)
#     print(f" MEGC EVALUATION | Task: {classify_question(question).upper()} ")
#     print("="*40)
#     print(f"Câu hỏi: {question}")
#     print(f"-> UF1 (Unweighted F1): {UF1:.4f}")
#     print(f"-> UAR (Unweighted Recall): {UAR:.4f}")
#     print("-" * 40)

#     print("\n[Per-class F1-Score]:")
#     for i, name in enumerate(class_names):
#         print(f"  + {name:<12}: {f1_per_class[i]:.4f}")

#     print("\n[Per-class Recall]:")
#     for i, name in enumerate(class_names):
#         print(f"  + {name:<12}: {recall_per_class[i]:.4f}")

#     return UF1, UAR